# Create Cancer Research Society Awards (Société de recherche sur le cancer, Canada)

Creates Cancer Research Society (CRS) **Operating Grants** from the competition
results PDFs CRS hosts on its own site
(`cancerresearchsociety.ca/content/uploads/2024/04/OG-Resultats-{year}.pdf`,
enumerated through the site's public WordPress media API). Each PDF is a table:
PI ("Family, Given"), institution, project title, amount awarded
("120 000 $/ 2 ans") and, for co-funded grants, a "Financé en partenariat avec
..." note naming the partner. **669 grants, competitions 2015-2023, 100%
title/PI/amount, 99.9% institution, CAD 80.6M** (local run 2026-10-01; per-year
counts match the totals printed on each PDF's cover, e.g. 74 in 2019, 83 in 2023).

**Language:** the PDFs on CRS's own site are the French versions, so titles are
mostly French (a few were submitted in English). The English versions exist only
as app.box.com shared links.

**Coverage gap (not fetched):** the 2024 and 2025 Operating Grant results, the
Doctoral Research Awards (2023-2025), the OCC/OvCAN partnership grants (2021) and
UPCYCLE (2018, 2022) are published only as app.box.com shared-file links; Box's
robots.txt disallows its download endpoint for all user agents, so they are not
ingested. The handful of single partnership grants described in prose on the
programme pages (Bridging the Gap, DECRYPT, 100 Years) are not tabulated and are
also left out.

**Co-funding:** 304 grants (45%) carry a partner note; it is kept in
`funder_scheme` ("Operating Grants (co-funded: {partner})"). 108 are co-funded
with CIHR institutes (mostly the Institute of Cancer Research); CIHR's own open
data also lists its share of these under F4320334506 as "CIHR - ICR / Cancer
Research Society Partnership - Research Grants" (CIHR funding reference numbers,
provenance `cihr_opendata`). They are kept here under CRS as distinct awards from
a distinct funder, and flagged.

**Prerequisites:**
- Run `scripts/local/cancer_research_society_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/cancer_research_society/cancer_research_society_projects.parquet`
  (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** synthetic and stable,
`CRS-OG-{competition year}-{family-given slug}` (a PI with two grants in one
competition, 5 cases, gets a 6-hex title hash appended). Works cite CRS's own
5-digit grant numbers (507 of the 1,212 stub rows on F4320311169 look like
`22504`), but the results PDFs do not print them, so 0 stubs collapse.

**Dates:** the PDFs give the competition year and the duration ("/ 2 ans"):
`start_date` = 1 January of the competition year (year-only precedent),
`end_date` NULL; `duration_years` is kept in the raw table.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320311169`
- display_name: Cancer Research Society
- ror_id / doi: from `openalex.funders.funders` (ror 00t38a349, doi 10.13039/100009326)

**Priority:** `565` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cancer_research_society_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cancer_research_society/cancer_research_society_projects.parquet`;

In [ ]:
%sql
SELECT competition_year, COUNT(*) as grants, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) as total_cad
FROM openalex.awards.cancer_research_society_raw
GROUP BY competition_year ORDER BY competition_year;

In [ ]:
%sql
DESCRIBE openalex.awards.cancer_research_society_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.cancer_research_society_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320311169 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320311169;

## Step 2: Create Cancer Research Society Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cancer_research_society_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320311169  -- Cancer Research Society
),

g AS (
    SELECT
        *,
        TRY_CAST(competition_year AS INT) AS yr,
        NULLIF(TRIM(partner), '') AS partner_clean
    FROM openalex.awards.cancer_research_society_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'CAD' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CASE WHEN g.partner_clean IS NOT NULL
         THEN CONCAT('Operating Grants (co-funded: ', g.partner_clean, ')')
         ELSE 'Operating Grants' END as funder_scheme,
    'crs_operating_grant_results' as provenance,
    CASE WHEN g.yr IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.competition_year, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.yr as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.source_pdf as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'crs_operating_grant_results' AND priority = 565;

-- Insert into openalex_awards_raw with priority.
-- Priority 565: direct-from-funder ingest of CRS's own competition results.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    565 as priority
FROM openalex.awards.cancer_research_society_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids, COUNT(DISTINCT funder_award_id) as distinct_award_ids
FROM openalex.awards.cancer_research_society_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is the synthetic CRS-OG-{year}-{slug}[-{hash}] form.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^CRS-OG-20[0-9]{2}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as well_formed
FROM openalex.awards.cancer_research_society_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.cancer_research_society_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.cancer_research_society_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.cancer_research_society_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 30;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.cancer_research_society_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.cancer_research_society_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/amount/PI, CAD only, amounts ~57K-150K).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.cancer_research_society_awards;

In [ ]:
%sql
-- Overlap with the CIHR open-data ingest: CIHR-ICR / CRS partnership grants that
-- CIHR also reports (its share, under its own FRN). Same PI family name + competition
-- year (CIHR fiscal year starting in, or one after, the CRS competition year).
SELECT COUNT(DISTINCT a.funder_award_id) as crs_awards_also_in_cihr
FROM openalex.awards.cancer_research_society_awards a
JOIN openalex.awards.cihr_raw c
  ON LOWER(TRIM(c.family_name)) = LOWER(a.lead_investigator.family_name)
 AND LOWER(c.program_name) LIKE '%cancer research society%'
 AND TRY_CAST(SUBSTR(c.competition_fiscal_year, 1, 4) AS INT) BETWEEN a.start_year - 1 AND a.start_year + 1;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'crs_operating_grant_results'
GROUP BY provenance, priority;